# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "A cookie has no soul, it’s just a cookie. But before, it was milk and eggs. And within the eggs lies the potential for life."
print(sample_sentence)


c:\Users\avigd\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\avigd\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\avigd\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an adminis

A cookie has no soul, it’s just a cookie. But before, it was milk and eggs. And within the eggs lies the potential for life.


In [2]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | a            |  1037
    2 | cookie       | 17387
    3 | has          |  2038
    4 | no           |  2053
    5 | soul         |  3969
    6 | ,            |  1010
    7 | it           |  2009
    8 | ’            |  1521
    9 | s            |  1055
   10 | just         |  2074
   11 | a            |  1037
   12 | cookie       | 17387
   13 | .            |  1012
   14 | but          |  2021
   15 | before       |  2077
   16 | ,            |  1010
   17 | it           |  2009
   18 | was          |  2001
   19 | milk         |  6501
   20 | and          |  1998
   21 | eggs         |  6763
   22 | .            |  1012
   23 | [SEP]        |   102

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Special tokens (index, token): [(0, '[CLS]'), (23, '[SEP]')]


### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.
- TODO: Explain how the attention mask hides padded positions from self-attention.


## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [4]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# sentence = "There's a space kept in hell with your name on the seat. With a spike in the chair just to make it complete. When you look at yourself do you see what I see. If you do why the fuck are you looking at me."
sentence = "There's a look on your face I would like to knock out. See the sin in your grin and the shape of your mouth. All I want is to see you in terrible pain. Though we won't ever meet I remember your name. Can't believe you were once just like anyone else. then you grew and became like the devil himself. Pray to God I think of a nice thing to say. But I don't think I can so fuck you anyway. You`re a scum, you`re a scum and I hope that you know. That the cracks in your smile are beginning to show. Now the world needs to see that it's time you should go. There's no light in your eyes and your brain is too slow. Can't believe you were once just like anyone else. then you grew and became like the devil himself. Pray to God I can think of a nice thing to say. But I don't think I can so fuck you anyway. Bet you sleep like a child with your thumb in your mouth. I could creep up beside put a gun in your mouth. makes me sick when I hear all the shit that you say. so much crap coming out it must take you all day. There's a space kept in hell with your name on the seat. With a spike in the chair just to make it complete. When you look at yourself do you see what I see. If you do why the fuck are you looking at me. There's a time for us all and I think yours has been. Can you please hurry up cos I find you obscene. We can't wait for the day that you're never around. When that face isn't here and you rot underground. Can't believe you were once just like anyone else. Then you grew and became like the devil himself. Pray to god I can think of a nice thing to say. But I don't think I can so fuck you anyway. So fuck you anyway. "
prediction = sentiment_pipeline(sentence)
prediction


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1615.57it/s]


[{'label': 'NEGATIVE', 'score': 0.9977678060531616}]

### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?
- TODO: How confident is the model and what does the score tell you?


## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        text = text.strip()
        encoding = self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {k: v.to(self.device) for k, v in encoding.items()}

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        predicted_id = torch.argmax(probs).item()
        label = self.model.config.id2label[predicted_id]
        return {"label": label, "probability": probs[predicted_id].item()}


In [8]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()
samples = [
    "There's a look on your face I would like to knock out. See the sin in your grin and the shape of your mouth. All I want is to see you in terrible pain. Though we won't ever meet I remember your name. Can't believe you were once just like anyone else. then you grew and became like the devil himself. Pray to God I think of a nice thing to say. But I don't think I can so fuck you anyway. You`re a scum, you`re a scum and I hope that you know. That the cracks in your smile are beginning to show. Now the world needs to see that it's time you should go. There's no light in your eyes and your brain is too slow. Can't believe you were once just like anyone else. then you grew and became like the devil himself. Pray to God I can think of a nice thing to say. But I don't think I can so fuck you anyway. Bet you sleep like a child with your thumb in your mouth. I could creep up beside put a gun in your mouth. makes me sick when I hear all the shit that you say. so much crap coming out it must take you all day. There's a space kept in hell with your name on the seat. With a spike in the chair just to make it complete. When you look at yourself do you see what I see. If you do why the fuck are you looking at me. There's a time for us all and I think yours has been. Can you please hurry up cos I find you obscene. We can't wait for the day that you're never around. When that face isn't here and you rot underground. Can't believe you were once just like anyone else. Then you grew and became like the devil himself. Pray to god I can think of a nice thing to say. But I don't think I can so fuck you anyway. So fuck you anyway.",
    "I’ll build the coziest lantern. To help you cross the ocean. I want to keep you dry. I want to keep the spark. And if the storm is hitting. I’ll try to keep you steady. And if I feel I‘m shaking. I would return to practice. And if my arm is hurting. And I if I need to rest. We’d have to switch positions. And you would do it well. We’d light the darkest forest. Brighten our darkest sins. We’d light the deepest cave and. Enlight the deepest meanings. You know I gave my breath so I could feed it. I blew too hard I thought I killed it. I saw an ember hidden in the ashes. I’ll blow again if it can be ignited. I’d build the strongest walls. (So you could feel protected). Made of the purest glass. (Cause you simply deserve it). I‘ll build the coziest home. (And we would learn to live in). Made of the purest love. (Cause you simply deserve it). And you could hear the rain banging the ceiling. And not a single drop could stain our feelings. And you could hear the rain banging the ceiling. And not a single drop could stain our feelings. You gave me your trust cause you knew I could keep it. The fire’s so pure we could learn how to drink it. You gave me your heart cause you knew I could light it. Cause you know, cause you know. Cause you know, cause you know. Cause you know, cause you know. Oh, cause you know, cause you know."
]
for text in samples:
    print(text)
    print(analyzer.predict(text))


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 725.00it/s]


There's a look on your face I would like to knock out. See the sin in your grin and the shape of your mouth. All I want is to see you in terrible pain. Though we won't ever meet I remember your name. Can't believe you were once just like anyone else. then you grew and became like the devil himself. Pray to God I think of a nice thing to say. But I don't think I can so fuck you anyway. You`re a scum, you`re a scum and I hope that you know. That the cracks in your smile are beginning to show. Now the world needs to see that it's time you should go. There's no light in your eyes and your brain is too slow. Can't believe you were once just like anyone else. then you grew and became like the devil himself. Pray to God I can think of a nice thing to say. But I don't think I can so fuck you anyway. Bet you sleep like a child with your thumb in your mouth. I could creep up beside put a gun in your mouth. makes me sick when I hear all the shit that you say. so much crap coming out it must take 

## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        '''TODO: load the tokenizer and model, and detect the available device.'''
        raise NotImplementedError("Initialize tokenizer, model, and device.")

    def recognize(self, text: str):
        '''TODO: tokenize the text, run the model, map predictions to BIO labels, and merge word pieces.'''
        raise NotImplementedError("Return structured entities.")


In [ ]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.
# ner = BERTNamedEntityRecognizer()
# sample_text = "TODO: add a short paragraph with at least two entities."
# ner.recognize(sample_text)


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | TODO | TODO |
| Primary purpose | TODO | TODO |
| Typical use cases | TODO | TODO |
| Strengths | TODO | TODO |
| Weaknesses | TODO | TODO |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. TODO: Describe how BERT encodes queries and documents.
2. TODO: Explain how those embeddings are stored and searched in a vector database.
3. TODO: Outline how the retrieved passages are handed to a generative model like GPT.
4. TODO: Provide a concrete application example (industry or product) where RAG with BERT makes sense.
